# 03 — Feature Engineering

Four match-level features were engineered and added to `combined_df`:

| Feature | Description |
|---------|-------------|
| `home_win_rate_last3` / `home_win_rate_last5` | OHL win rate over the last 3 and 5 home games |
| `ohl_season_points` | OHL's cumulative points in the season at the time of the match |
| `opp_ppg_vs_ohl` | Opponent's historical points-per-game against OHL (proxy for opponent strength) |
| `rolling_avg_attendance_last3` | Average actual attendance over the last 3 home games |

All features were computed with a **shift to exclude the current match**, preventing data leakage.

In [1]:
import pandas as pd

combined_df = pd.read_csv('../data/cleaned/combined_df.csv')
combined_df['match_date'] = pd.to_datetime(combined_df['match_date'])

# Load ALL OHL matches (home + away) — needed for form and season points
raw_match = pd.read_csv('../data/raw/gold_match.csv')
raw_match['match_date'] = pd.to_datetime(raw_match['match_date'])
raw_match['is_home_match'] = raw_match['is_home_match'].astype(bool)

# Derive OHL result and points for every match
raw_match['opponent'] = raw_match.apply(
    lambda r: r['away_team'] if r['is_home_match'] else r['home_team'], axis=1
)
raw_match['ohl_result'] = raw_match.apply(
    lambda r: r['result_home'] if r['is_home_match']
    else {'W': 'L', 'L': 'W', 'D': 'D'}.get(r['result_home']),
    axis=1
)
raw_match['ohl_points'] = raw_match['ohl_result'].map({'W': 3, 'D': 1, 'L': 0})
raw_match = raw_match.sort_values('match_date').reset_index(drop=True)

print(f"All OHL matches: {len(raw_match)}")
raw_match[['match_date', 'opponent', 'is_home_match', 'ohl_result', 'ohl_points']].head(8)

All OHL matches: 142


,match_date,opponent,is_home_match,ohl_result,ohl_points
0,2022-07-23,Kortrijk,False,W,3
1,2022-07-30,Westerlo,True,W,3
2,2022-08-07,Antwerp,False,L,0
3,2022-08-14,Club Brugge,True,L,0
4,2022-08-21,Standard Liège,False,W,3
5,2022-08-27,KV Oostende,True,W,3
6,2022-09-04,Anderlecht,False,D,1
7,2022-09-10,Sporting Charleroi,True,W,3


## Feature 1 — Home form (win rate over last 3 and 5 home games)

In [2]:
# Use home matches only, sorted by date
home_only = raw_match[raw_match['is_home_match']].copy().reset_index(drop=True)
home_only['is_win'] = (home_only['ohl_result'] == 'W').astype(int)

# shift(1) excludes the current match result — no leakage
home_only['home_win_rate_last3'] = (
    home_only['is_win'].shift(1).rolling(3, min_periods=1).mean()
)
home_only['home_win_rate_last5'] = (
    home_only['is_win'].shift(1).rolling(5, min_periods=1).mean()
)

form_df = home_only[['match_id', 'home_win_rate_last3', 'home_win_rate_last5']]

print(form_df.head(8))
print(f"\nNulls: {form_df.isnull().sum().sum()}  (none expected — min_periods=1 fills early rows)")

                    match_id  home_win_rate_last3  home_win_rate_last5
0  d256yo3eng04m0fu7b4sl7wno                  NaN                  NaN
1  d4mn5ksbxuvnaww4pmommxhqs             1.000000             1.000000
2  d65hmi7sq03yzr5he1k7ypus4             0.500000             0.500000
3  d80mkemezkz16bqh6lbn8tlhw             0.666667             0.666667
4  dak40etbhbqsr1nxyt50qcg0k             0.666667             0.750000
5  dcl81t8chgq07o8rw8orj5no4             0.666667             0.600000
6  dfui3aovgvfccasonp6w5ajh0             0.333333             0.400000
7  di0x0as1g324k0hoea1o5vvh0             0.000000             0.400000

Nulls: 2  (none expected — min_periods=1 fills early rows)


## Feature 2 — OHL season points at time of match (table position proxy)

Uses all OHL matches (home + away). For each match, cumulative points earned in that season **before** this match. A true league table would require all teams' results — unavailable here, so season points is the best proxy from the data we have.

In [3]:
# Sort all matches by season then date
all_matches = raw_match.sort_values(['season', 'match_date']).copy()

# Cumulative points within each season, shifted by 1 to exclude current match
all_matches['ohl_season_points'] = (
    all_matches.groupby('season')['ohl_points']
    .transform(lambda x: x.shift(1).cumsum().fillna(0))
)

# Keep only home matches for merging
season_points_df = all_matches[all_matches['is_home_match']][['match_id', 'ohl_season_points']]

print(season_points_df.head(8))

                     match_id  ohl_season_points
1   d256yo3eng04m0fu7b4sl7wno                3.0
3   d4mn5ksbxuvnaww4pmommxhqs                6.0
5   d65hmi7sq03yzr5he1k7ypus4                9.0
7   d80mkemezkz16bqh6lbn8tlhw               13.0
9   dak40etbhbqsr1nxyt50qcg0k               17.0
11  dcl81t8chgq07o8rw8orj5no4               20.0
14  dfui3aovgvfccasonp6w5ajh0               21.0
16  di0x0as1g324k0hoea1o5vvh0               22.0


## Feature 3 — Opponent strength proxy (points-per-game against OHL)

A true opponent ranking requires all teams' results against all opponents — not available. Instead: for each home match, compute the opponent's **historical points-per-game in all previous matches against OHL** (opponent W=3, D=1, L=0). Higher value = tougher opponent.

In [4]:
# Opponent points: inverse of OHL result
raw_match['opp_points'] = raw_match['ohl_result'].map({'W': 0, 'D': 1, 'L': 3})

# Sort by opponent + date, compute expanding mean of opp_points shifted to exclude current match
opp_sorted = raw_match.sort_values(['opponent', 'match_date']).copy()
opp_sorted['opp_ppg_vs_ohl'] = (
    opp_sorted.groupby('opponent')['opp_points']
    .transform(lambda x: x.shift(1).expanding().mean())
)

# For first ever meeting there is no history — fill with overall average opponent PPG
overall_avg = raw_match['opp_points'].mean()
opp_sorted['opp_ppg_vs_ohl'] = opp_sorted['opp_ppg_vs_ohl'].fillna(overall_avg)

# Keep only home matches for merging
opp_strength_df = opp_sorted[opp_sorted['is_home_match']][['match_id', 'opp_ppg_vs_ohl']]

print(opp_strength_df.describe())
print(f"\nNulls after fill: {opp_strength_df['opp_ppg_vs_ohl'].isnull().sum()}")

       opp_ppg_vs_ohl
count       71.000000
mean         1.440617
std          0.823374
min          0.000000
25%          1.000000
50%          1.500000
75%          2.000000
max          3.000000

Nulls after fill: 0


## Merge all features into combined_df

## Feature 4 — Rolling average home attendance (last 3 games)\n\nThe strongest single predictor of attendance (Spearman r = 0.475). If fans have been showing\nup to recent home games, they are likely to come to the next one too — capturing fan momentum\nand general season enthusiasm that other features miss.\n\nComputed on home matches only, shifted by 1 to exclude the current match.

In [5]:
# Use home matches sorted by date — same base as home form features
# tickets_scanned is available from combined_df via the match_id
home_attendance = combined_df[combined_df['match_id'].isin(
    home_only['match_id']
)][['match_id', 'tickets_scanned']].copy()

# Merge date order from home_only so we can sort correctly
home_attendance = home_attendance.merge(
    home_only[['match_id', 'match_date']], on='match_id'
).sort_values('match_date').reset_index(drop=True)

# shift(1) excludes current match — no leakage
home_attendance['rolling_avg_attendance_last3'] = (
    home_attendance['tickets_scanned']
    .shift(1)
    .rolling(3, min_periods=1)
    .mean()
)

attendance_roll_df = home_attendance[['match_id', 'rolling_avg_attendance_last3']]

print(attendance_roll_df.head(8))
print(f"\nNulls: {attendance_roll_df['rolling_avg_attendance_last3'].isnull().sum()}  (1 expected — first match has no history)")

                    match_id  rolling_avg_attendance_last3
0  d256yo3eng04m0fu7b4sl7wno                           NaN
1  d4mn5ksbxuvnaww4pmommxhqs                   5565.000000
2  d65hmi7sq03yzr5he1k7ypus4                   6502.500000
3  d80mkemezkz16bqh6lbn8tlhw                   5831.333333
4  dak40etbhbqsr1nxyt50qcg0k                   5479.000000
5  dcl81t8chgq07o8rw8orj5no4                   5095.666667
6  dfui3aovgvfccasonp6w5ajh0                   6012.000000
7  di0x0as1g324k0hoea1o5vvh0                   6898.000000

Nulls: 1  (1 expected — first match has no history)


In [6]:
engineered_df = (
    combined_df
    .merge(form_df,            on='match_id', how='left')
    .merge(season_points_df,   on='match_id', how='left')
    .merge(opp_strength_df,    on='match_id', how='left')
    .merge(attendance_roll_df, on='match_id', how='left')
)

print(f"Shape: {engineered_df.shape}")
print(f"\nNew features:")
print(engineered_df[['match_date', 'opponent', 'home_win_rate_last3',
                      'ohl_season_points', 'opp_ppg_vs_ohl',
                      'rolling_avg_attendance_last3']].head(10))

Shape: (71, 36)

New features:
  match_date              opponent  home_win_rate_last3  ohl_season_points  \
0 2022-07-30              Westerlo                  NaN                3.0   
1 2022-08-14           Club Brugge             1.000000                6.0   
2 2022-08-27           KV Oostende             0.500000                9.0   
3 2022-09-10    Sporting Charleroi             0.666667               13.0   
4 2022-10-01  Union Saint-Gilloise             0.666667               17.0   
5 2022-10-15                  Genk             0.666667               20.0   
6 2022-10-30                  Gent             0.333333               21.0   
7 2022-11-11               Seraing             0.000000               22.0   
8 2023-01-08              Kortrijk             0.333333               26.0   
9 2023-01-17                 Eupen             0.333333               29.0   

   opp_ppg_vs_ohl  rolling_avg_attendance_last3  
0        1.521127                           NaN  
1        1

## Opponent grouping — rare opponents → "Other"

One-hot encoding all 21 opponents would cost 20 degrees of freedom on only 71 rows.
Opponents appearing **only once** cannot have a reliable coefficient estimated, so they
are grouped into a single `"Other"` category, reducing the encoding to 16 dummies.

In [7]:
# Opponents with only 1 home appearance — cannot estimate a reliable coefficient
rare_opponents = (
    engineered_df['opponent'].value_counts()
    [lambda s: s == 1]
    .index.tolist()
)

engineered_df['opponent_grouped'] = engineered_df['opponent'].where(
    ~engineered_df['opponent'].isin(rare_opponents), other='Other'
)

print(f"Rare opponents grouped into 'Other': {rare_opponents}")
print(f"\nopponent_grouped value counts:")
print(engineered_df['opponent_grouped'].value_counts())

Rare opponents grouped into 'Other': ['KV Oostende', 'Seraing', 'RWDM', 'Beerschot', 'RAAL La Louvière']

opponent_grouped value counts:
opponent_grouped
Westerlo                6
Mechelen                6
Standard Liège          6
Other                   5
Sporting Charleroi      5
Gent                    5
Sint-Truiden            5
Club Brugge             4
Union Saint-Gilloise    4
Genk                    4
Cercle Brugge           4
Anderlecht              4
Kortrijk                3
Antwerp                 3
Dender                  3
Eupen                   2
Zulte Waregem           2
Name: count, dtype: int64


In [8]:
engineered_df.to_csv('../data/cleaned/engineered_df.csv', index=False)
print('Saved.')

Saved.


## 5 — Additional Derived Features

Four new features engineered from columns already in the dataset:

| Feature | Formula | Why |
|---|---|---|
| `matchday_normalized` | `matchday / max_matchday_in_season` | Captures season phase (0→1); late-season games draw +40% more fans |
| `last_h2h_goal_margin` | `last_h2h_ohl_goals − last_h2h_opponent_goals` | Richer than W/D/L — a 4-0 win signals different fan anticipation than a 1-0 |
| `ohl_points_per_game` | `ohl_season_points / matchday` | Normalises form rate; raw cumulative points inflate late in season |
| `pre_match_interest_ratio` | `avg_ohl_interest_7d / season_mean` | Within-season relative buzz; Google Trends resets between seasons |

In [ ]:
# Re-load the saved CSV so this section is independently runnable
engineered_df = pd.read_csv('../data/cleaned/engineered_df.csv')
engineered_df['match_date'] = pd.to_datetime(engineered_df['match_date'])

# --- matchday_normalized ---
# BUG FIX: do NOT use transform('max') per season — the 2025/26 season is incomplete
# in our data (last home match = matchday 28), so its per-season max would be 28
# while complete training seasons have a true final matchday of ~33.
# This would inflate all 2025/26 values (matchday 20 → 20/28=0.71 vs 20/33=0.61),
# making the model see the test season as perpetually "late season".
# Fix: use the max matchday from training seasons only as a fixed denominator.
TRAIN_SEASONS = ['2022/2023', '2023/2024', '2024/2025']
fixed_season_length = int(engineered_df[engineered_df['season'].isin(TRAIN_SEASONS)]['matchday'].max())
engineered_df['matchday_normalized'] = engineered_df['matchday'] / fixed_season_length
print(f"Fixed season length (training max matchday): {fixed_season_length}")

# --- last_h2h_goal_margin ---
engineered_df['last_h2h_goal_margin'] = (
    engineered_df['last_h2h_ohl_goals'].fillna(0)
    - engineered_df['last_h2h_opponent_goals'].fillna(0)
)

# --- ohl_points_per_game ---
engineered_df['ohl_points_per_game'] = (
    engineered_df['ohl_season_points'] / engineered_df['matchday']
)

# --- pre_match_interest_ratio ---
# BUG FIX: do NOT use transform('mean') — it computes the full-season mean
# simultaneously, so the July 2025 match (matchday 1) ends up using March 2026
# interest values in its denominator. This leaks future information into every
# early-season match, in both training and test seasons.
# Fix: use an expanding within-season mean shifted by 1, so each match only
# sees interest data from previous matches in that season.
df_s = engineered_df.sort_values(['season', 'match_date']).copy()
expanding_mean = df_s.groupby('season')['avg_ohl_interest_7d'].transform(
    lambda x: x.shift(1).expanding().mean()
)
df_s['pre_match_interest_ratio'] = df_s['avg_ohl_interest_7d'] / expanding_mean
# First match of each season has no prior data → fill with 1.0 (neutral: interest = baseline)
df_s['pre_match_interest_ratio'] = df_s['pre_match_interest_ratio'].fillna(1.0)
engineered_df = df_s.sort_values('match_date').reset_index(drop=True)

engineered_df.to_csv('../data/cleaned/engineered_df.csv', index=False)

print(f'Saved. Shape: {engineered_df.shape}')
print('\nNew features sample:')
print(engineered_df[[
    'match_date', 'opponent', 'matchday', 'matchday_normalized',
    'last_h2h_goal_margin', 'ohl_points_per_game', 'pre_match_interest_ratio'
]].head(8).to_string())